# Reproducing Sohn et al. (2019) from dandiset 000130

Paper. Sohn H, Narain D, Meirhaeghe N, Jazayeri M (2019). Bayesian computation through cortical
latent dynamics. *Neuron* 103(5):934-947.
[doi:10.1016/j.neuron.2019.06.012](https://doi.org/10.1016/j.neuron.2019.06.012). This attempt used
the full text of the accepted author manuscript in PubMed Central (PMC6805134). The supplementary
figures and tables (Tables S1 to S3) were not available to it.

Dandiset. [DANDI:000130](https://dandiarchive.org/dandiset/000130/0.241017.1448), version
0.241017.1448. The paper cites no DANDI version (it predates the dandiset, which was made for the
Neural Latents Benchmark in 2021). Of the two published versions, 0.220113.0407 and 0.241017.1448,
this is the most recent. Both hold the same two files (same asset and blob ids and sizes), and
their metadata differ only in the publication date, the citation and the repository URL's trailing
slash, so neither matches the paper better.

What the dandiset holds. One recording session (2016-12-11) of monkey H ("Haydn"), from the
Neural Latents Benchmark (NLB): a train file with the full trial table and the spikes of 54 units,
and a test file with 283 further trials of the same session that carry no task variables and only
40 of the units. The paper analyses 17 sessions of monkey H and 12 of monkey G. Monkey G is not in
the dandiset. All results here come from this one session, and a "dataset" is one effector and
target direction (4 here; the paper's 8 are 2 animals x 4).

Code. The paper's code is "available from the corresponding author on request" and was not
read. The dandiset links the NLB, whose code
([nlb_tools](https://github.com/neurallatents/nlb_tools) at commit 42f8410b) loads these files,
builds tensors aligned to Go (1500 ms before Go), and scores models by a trial-by-trial correlation
of neural speed with tp within each prior, effector and direction. It holds none of the paper's
analyses; its speed measure (mean step between successive states) matches the one chosen here.

| Claim | Source | Tested here |
|---|---|---|
| beh-slopes: slopes of tp on ts between 0 and 1 | Fig. 1E | Tested |
| beh-overlap-prior: opposite biases at ts = 800 ms | Fig. 1E | Tested |
| beh-sigmoid-increments: smaller tp increments at the extremes | Fig. 2F | Tested |
| beh-long-more-bias: smaller slopes for the Long prior | Fig. 2H | Tested |
| beh-bayes-model: the Bayesian observer captures bias and variance | Fig. 2C, D | Tested |
| glm-prior-ts: 65% prior-dependent and 27% ts-dependent units (H) | Fig. 3E | Tested |
| dfr-overlap-support: prior difference at 800 ms peaks during the support | Fig. 3D | Tested |
| pca-dimensionality: 3-4 PCs (estimation), 5-10 (production) | Fig. S3 | Tested |
| encoding-r2: projections match the Bayesian estimate (R2 0.993, 0.996) | Fig. 4C | Tested |
| encoding-bayes-beats-linear: Bayesian better than linear | Fig. 4D | Tested |
| short-in-long-control: no sigmoid in the matched Long period | Fig. 4D | Tested |
| neural-long-more-bias: smaller neural slope for Long | Fig. 4D | Tested |
| speed-vs-projection: larger projection, slower speed | Fig. 4E | Tested |
| initial-speed-biased: initial speed biased toward the mean | Fig. 4F | Tested |
| speed-predicts-tp: speed predicts tp | Fig. 4G | Tested |
| alt-speed-unmodulated: speed constant during the support | Fig. 5B | Tested |
| alt-transient-parallel: parallel trajectories after Set | Fig. 5D | Tested |
| alt-threshold-monotonic: distances fall steadily up to Go | Fig. 5F | Tested |
| gpfa-variance-inverted-u: inverted-U variance of single-trial projections | Fig. 6C | Tested |
| gpfa-mean-sigmoid: sigmoidal mean, higher Short slope | Fig. 6D | Tested |
| rnn-curved-manifold, rnn-perturbation: recurrent network models | Fig. 7 | Blocked (no model in the dandiset; code on request) |
| monkey-g: the results hold in monkey G | throughout | Blocked (monkey G not in the dandiset) |

In [ ]:
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

import rsg

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
data = rsg.load()
S, T = data["train"], data["test"]
trials, spikes = S["trials"], S["spikes"]
good = rsg.completed(trials)
units, unit_table = rsg.select_units(S, good)
DS = [(e, th, rsg.dataset_name(e, th)) for e, th in rsg.DATASETS]
print(f"loaded dandiset {rsg.DANDISET_ID} version {rsg.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
for key, f in [("train", S), ("test", T)]:
    print(f"{key}: {data['urls'][rsg.TRAIN if key == 'train' else rsg.TEST]}")
    for k, v in f["meta"].items():
        print(f"   {k}: {v}")
    print(f"   trials: {len(f['trials'])}; columns: {list(f['trials'].columns)}")
    print(f"   units: {len(f['unit_ids'])}; held out: {int(f['heldout'].sum())}")
print("\nTest-file units that are in the train file:", int(np.isin(T["unit_ids"], S["unit_ids"]).sum()), "of", len(T["unit_ids"]),
      "; train units held out:", int(S["heldout"].sum()), "; held-out train units in the test file:",
      int(np.isin(S["unit_ids"][S["heldout"]], T["unit_ids"]).sum()))
print("\nTrain trials by NLB split:", trials.split.value_counts().to_dict())
print("Train trials with Set and Go:", int((trials.set_time.notna() & trials.go_time.notna()).sum()),
      "; of these flagged is_outlier:", int(trials[trials.set_time.notna() & trials.go_time.notna()].is_outlier.sum()),
      "; completed and not outliers (used here):", len(good))
print("Used trials by split:", good.split.value_counts().to_dict())
print("\nUsed trials per condition (rows: effector, direction, prior; columns: ts in ms):")
print(pd.crosstab([good.is_eye.map({True: 'Eye', False: 'Hand'}), good.theta.map({0.0: 'Right', 180.0: 'Left'}),
                   good.is_short.map({True: 'Short', False: 'Long'})], good.ts).to_string())
print("\nColumn descriptions (train trial table):")
for c, d in S["descriptions"].items():
    print(f"   {c}: {d}")

The dandiset summary says one subject (Rhesus monkey) and 2 files, which the files confirm. The
test file holds 283 trials of the same session (`session_id` 20161211) with only `start_time`,
`stop_time`, `go_time` and `split`, on its own clock (0 to 509 s), and only the 40 held-in units;
its task variables and the 14 held-out units are withheld for the benchmark. These trials cannot
enter any analysis that needs ts, the prior or the effector, so the analyses use the train file's
trials. Each condition has at least 14 used trials, so the Methods' rule of at least 5 trials per
condition does not remove any unit.

In [ ]:
c = trials[trials.set_time.notna() & trials.go_time.notna()].copy()
d_ts = c.ts - (c.set_time - c.ready_time) * 1000
d_tp = c.tp - (c.go_time - c.set_time) * 1000
print("ts minus (set_time - ready_time), ms: median %.1f, range %.1f to %.1f; within 5 ms: %.0f%%"
      % (d_ts.median(), d_ts.min(), d_ts.max(), 100 * (abs(d_ts) <= 5).mean()))
for flag, g in d_tp.groupby(c.is_outlier):
    print(f"tp minus (go_time - set_time), ms, is_outlier={flag}: median {g.median():.1f}, 5th to 95th percentile "
          f"{g.quantile(.05):.1f} to {g.quantile(.95):.1f}, range {g.min():.1f} to {g.max():.1f}")
print("Distinct ts by prior:", {('Short' if k else 'Long'): sorted(v) for k, v in trials.groupby('is_short').ts.unique().items()})
print("iti by prior (ms):", {('Short' if k else 'Long'): sorted(v) for k, v in trials.groupby('is_short').iti.unique().items()})
print("Paper: inter-trial intervals of 1220 ms (Short) and 500 ms (Long).")
print("theta values:", sorted(trials.theta.unique()), "; is_eye values:", sorted(trials.is_eye.unique()))
print("tp of used trials, ms: %.0f to %.0f; of uncompleted trials (no Go):" % (good.tp.min(), good.tp.max()),
      sorted(trials[trials.go_time.isna()].tp.round().unique())[:6], "...")

# Refit of the Methods' mixture model, per prior, ts, effector and direction, on completed trials.
flags = []
for key, g in c[c.tp > 0].groupby(["is_short", "ts", "is_eye", "theta"]):
    r = rsg.outlier_mixture(g.tp.to_numpy(), key[1])
    flags.append(pd.Series(r > 0.5, index=g.index))
refit = pd.concat(flags).reindex(c.index, fill_value=True)
print("\nMixture-model refit against is_outlier (completed trials):")
print(pd.crosstab(c.is_outlier.rename("is_outlier"), refit.rename("refit lapse")).to_string())
print(f"Outlier fraction of completed trials: file {c.is_outlier.mean():.2%}, refit {refit.mean():.2%}; paper, monkey H: 3.84%")
only_file = c[c.is_outlier & ~refit]
print("tp / ts of trials flagged by the file but kept by the refit: %.2f to %.2f (%d trials); of trials flagged by both: %s"
      % ((only_file.tp / only_file.ts).min(), (only_file.tp / only_file.ts).max(), len(only_file),
         np.round(np.sort((c[c.is_outlier & refit].tp / c[c.is_outlier & refit].ts).to_numpy()), 2).tolist()))

`ts` is the nominal interval; the event times give intervals within 23.5 ms of it, consistent with
a 60 Hz display (one frame is 16.7 ms). `tp` is a median 18 ms longer than `go_time - set_time`
for used trials, so the two columns do not measure the same interval; the analyses use `tp` for
behaviour, as the trial table defines it, and the event times for aligning spikes. The `iti`
values agree with the paper (1220 ms after Short trials, 500 ms after Long ones), and its column
description (500 for Short, 1220 for Long) has them the other way round. The `is_outlier` flag
marks 46 completed trials (4.47%, against 3.84% for monkey H in the paper). A refit of the
Methods' mixture model as read here, per prior, ts, effector and direction, flags only 10 of them
and no other trial; the 36 others have tp between the ratios to ts printed above, which the refit
keeps as part of the Gaussian component. How the flag was computed is not documented beyond "probabilistic mixture model". The
analyses use the file's flag, as the claims file fixes.

In [ ]:
print("Units: rate over the observed intervals, spikes/s")
print(unit_table.rate.describe().round(2).to_string())
print(f"Kept by the Methods' rule (>= {rsg.MIN_RATE} spike/s and >= {rsg.MIN_TRIALS} trials per condition): "
      f"{len(units)} of {len(unit_table)} (held-out units among the kept: {int(unit_table.heldout[unit_table.kept].sum())})")
print("Paper: GPFA on 'simultaneously recorded neurons in a single session (N = 48 in H)'.")
obs = S["obs_intervals"]
same = all(np.array_equal(o, obs[0]) for o in obs)
print("\nAll units share the same observation intervals:", same, "; intervals:", np.round(obs[0], 2).tolist())
gaps = obs[0][1:, 0] - obs[0][:-1, 1]
inside = [((obs[0][:, 0] <= a) & (obs[0][:, 1] >= b)).any() for a, b in zip(trials.start_time, trials.stop_time)]
print(f"Gaps between intervals: {np.round(gaps, 3).tolist()} s; trials inside one interval: {sum(inside)} of {len(trials)}")
print("Spike times sorted within every unit:", all(np.all(np.diff(s) >= 0) for s in spikes),
      "; spikes outside the observation intervals:",
      int(sum((~((s[:, None] >= obs[0][:, 0]) & (s[:, None] <= obs[0][:, 1])).any(1)).sum() for s in spikes)))

# Recordings deposited twice: does any test trial reappear in the train file? Compare the
# inter-spike-interval sequence of each test trial with the whole train recording, for 3 units.
hits, checked = 0, 0
for uid in T["unit_ids"][:3]:
    s_tr = np.round(np.diff(spikes[int(np.flatnonzero(S["unit_ids"] == uid)[0])]), 4)
    s_te = T["spikes"][int(np.flatnonzero(T["unit_ids"] == uid)[0])]
    for a, b in zip(T["trials"].start_time, T["trials"].stop_time):
        pat = np.round(np.diff(s_te[(s_te >= a) & (s_te < b)]), 4)
        if len(pat) < 3:
            continue
        checked += 1
        hits += any(np.array_equal(s_tr[i:i + len(pat)], pat) for i in np.flatnonzero(s_tr == pat[0]))
print(f"\nTest trials whose spike pattern recurs in the train file: {hits} of {checked} checked (3 units)")
seg = rsg.counts(spikes, units, good.ready_time.to_numpy(), 0, 400, 400)[:, 0, :]
dup = pd.DataFrame(seg).duplicated().sum()
print("Train trials with identical spike counts in all kept units over Ready to Ready + 400 ms:", int(dup))

The 48 units kept by the Methods' rate rule match the paper's count for the single session of
monkey H used for GPFA (Methods, N = 48), which is consistent with this being that session; the
dandiset does not say so. All units share six observation intervals separated by gaps of 0.1 to
0.14 s; every trial lies inside one interval. No test trial repeats a stretch of the train file,
and no two train trials are identical, so no recording is deposited twice. The units carry no
single-unit or multi-unit label, no electrode, and the electrode table has no positions or
impedances.

In [ ]:
# Do spikes share the clock of the trial table? Decode "before" against "after" an event from the
# population counts in 200 ms windows, with the event time shifted by an offset. A response
# locked to the event gives the best decoding at zero offset.
rng_clock = np.random.default_rng(1)
offsets = [-400, -200, -100, -50, 0, 50, 100, 200, 400]
rows = []
for event in ["ready_time", "set_time", "go_time"]:
    for off in offsets:
        t0 = good[event].to_numpy() + off / 1000
        pre = rsg.counts(spikes, units, t0, -200, 0, 200)[:, 0, :]
        post = rsg.counts(spikes, units, t0, 0, 200, 200)[:, 0, :]
        X = np.sqrt(np.vstack([pre, post]))
        y = np.r_[np.zeros(len(pre)), np.ones(len(post))]
        perm = rng_clock.permutation(len(y))
        acc = cross_val_score(LogisticRegression(max_iter=2000), X[perm], y[perm], cv=5).mean()
        rows.append({"event": event, "offset_ms": off, "accuracy": acc})
clock = pd.DataFrame(rows).pivot(index="offset_ms", columns="event", values="accuracy")
print("Cross-validated accuracy, before against after the event (chance 0.5):")
print(clock.round(3).to_string())
print("Offset of best decoding:", clock.idxmax().to_dict())

# Contrast between conditions: decode the prior from the counts in the 400 ms before Ready, with
# the trial labels as they are and shifted by one block of trials.
pre_ready = np.sqrt(rsg.counts(spikes, units, good.ready_time.to_numpy(), -400, 0, 400)[:, 0, :])
for shift in [0, 25, 50]:
    y = np.roll(good.is_short.to_numpy(), shift)
    perm = rng_clock.permutation(len(y))
    acc = cross_val_score(LogisticRegression(max_iter=2000), pre_ready[perm], y[perm], cv=5).mean()
    print(f"Prior decoded before Ready, labels shifted by {shift} trials: accuracy {acc:.3f}")

Decoding of "before against after" is best at an offset of +100 ms for Ready, Set and Go, and falls
for negative offsets (at -400 ms, 0.56, 0.68 and 0.73). A best offset of +100 ms is what a response
with a latency of about 100 ms after each event gives, since the "after" window then holds the
response. The prior is decoded from the 400 ms before Ready with accuracy 0.64 with the labels in
place, 0.56 and 0.52 with the labels shifted by 25 and 50 trials. This is consistent with the
spikes and the trial table sharing one clock in the train file; it does not prove it. The test
file could not be checked this way, as it has no task variables; it is not used.

## Behaviour

### beh-slopes

> Regression slopes of tp on ts are positive and less than one in every condition, so the animals
> estimated ts and their responses were biased toward the mean of the prior (Figure 1E).

In [ ]:
rows = []
for e, th, name in DS:
    for short in [True, False]:
        g = good[(good.is_eye == e) & (good.theta == th) & (good.is_short == short)]
        b, se, ci = rsg.weighted_slope(g.ts, g.tp)
        b_ols = np.polyfit(g.ts, g.tp, 1)[0]
        rows.append({"dataset": name, "prior": "Short" if short else "Long", "trials": len(g), "slope": b,
                     "ci_low": ci[0], "ci_high": ci[1], "slope_unweighted": b_ols})
slopes = pd.DataFrame(rows)
print(slopes.round(3).to_string(index=False))
inside = ((slopes.slope > 0) & (slopes.slope < 1)).sum()
strict = ((slopes.ci_low > 0) & (slopes.ci_high < 1)).sum()
print(f"\nSlopes between 0 and 1: {inside} of 8; with the 95% interval excluding 0 and 1: {strict} of 8")
print("Slope range: %.3f to %.3f" % (slopes.slope.min(), slopes.slope.max()))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), sharey=True)
for ax, (e, th, name) in zip(axes, DS):
    for short, col in [(True, "tab:red"), (False, "tab:blue")]:
        g = good[(good.is_eye == e) & (good.theta == th) & (good.is_short == short)]
        m = g.groupby("ts").tp.agg(["mean", "std"])
        ax.errorbar(m.index, m["mean"], m["std"], color=col, marker="o", capsize=2)
    ax.plot([450, 1250], [450, 1250], "k:", lw=1)
    ax.set_title(name)
    ax.set_xlabel("ts (ms)")
axes[0].set_ylabel("tp (ms), mean and SD")
plt.tight_layout()
plt.show()
print("Mean tp per prior and ts: %.0f to %.0f ms; all points within the axes."
      % (good.groupby(['is_eye', 'theta', 'is_short', 'ts']).tp.mean().min(), good.groupby(['is_eye', 'theta', 'is_short', 'ts']).tp.mean().max()))

**Result: not reproduced.** Six of the 8 slopes lie between 0 and 1 (0.50 to 0.90), and 5 of 8 have
95% intervals that exclude both 0 and 1. In the Short prior with the eye, the slopes are 1.11
(Eye Right, interval 1.01 to 1.21) and 1.01 (Eye Left), so in these two conditions this session
shows no regression toward the mean. All slopes are positive. The criterion needs all 8 between 0
and 1. The paper's slopes pool 17 sessions of monkey H and 12 of G; this is one session.

### beh-overlap-prior

> At the overlapping ts of 800 ms, biases were in opposite directions for the two priors (rank-sum
> test, p < 1e-43 in animal H over 17 sessions).

In [ ]:
o = good[good.ts == 800]
s_tp, l_tp = o[o.is_short].tp, o[~o.is_short].tp
p = stats.ranksums(s_tp, l_tp).pvalue
print(f"ts = 800 ms: Short n = {len(s_tp)}, median tp - 800 = {s_tp.median() - 800:.1f} ms; "
      f"Long n = {len(l_tp)}, median tp - 800 = {l_tp.median() - 800:.1f} ms; rank-sum p = {p:.2g}")
for e, th, name in DS:
    g = o[(o.is_eye == e) & (o.theta == th)]
    a, b = g[g.is_short].tp, g[~g.is_short].tp
    print(f"   {name}: Short {a.median() - 800:+.1f} (n={len(a)}), Long {b.median() - 800:+.1f} (n={len(b)}), "
          f"p = {stats.ranksums(a, b).pvalue:.2g}")

**Result: not reproduced.** At ts = 800 ms the median bias is +28.3 ms in the Short prior and +47.6 ms
in the Long prior, so both are positive and not of opposite sign; the rank-sum test gives p = 0.073
(93 and 107 trials). The Long prior gives the longer tp in 2 of 4 datasets, Hand Right (+29.7
against +96.8 ms, p = 0.00099) and Hand Left (-61.6 against -2.8 ms, p = 0.069), where the Short
bias is negative. In the eye datasets the Short tp is the longer (Eye Left +79.1 against +18.6 ms,
p = 0.051).

### beh-sigmoid-increments

> Increments of tp between neighbouring ts are smaller at the extremes of the prior than near its
> mean (signed-rank test, p < 1e-11 in animal H).

In [ ]:
rows = []
for e, th, name in DS:
    for short in [True, False]:
        g = good[(good.is_eye == e) & (good.theta == th) & (good.is_short == short)]
        ext, mid = rsg.increments(g.groupby("ts").tp.mean().loc[list(rsg.prior_ts(short))])
        rows.append({"dataset": name, "prior": "Short" if short else "Long", "extreme": ext, "middle": mid})
inc = pd.DataFrame(rows)
inc["middle_minus_extreme"] = inc.middle - inc.extreme
print(inc.round(1).to_string(index=False))
w = stats.wilcoxon(inc.middle, inc.extreme)
print(f"\nMiddle above extreme in {(inc.middle_minus_extreme > 0).sum()} of 8; median difference "
      f"{inc.middle_minus_extreme.median():.1f} ms; two-sided signed-rank p = {w.pvalue:.4f}")

**Result: reproduced.** The mean increment of tp between neighbouring ts is larger in the middle of
the prior than at its extremes in 7 of 8 conditions (median difference 31.3 ms), two-sided
signed-rank p = 0.0156. The exception is Hand Left, Short prior (-10.7 ms).

### beh-long-more-bias

> Biases are larger for the Long prior than for the Short prior; regression slopes are smaller for
> Long (Figure 2H).

In [ ]:
wide = slopes.pivot(index="dataset", columns="prior", values="slope")
wide["long_below_short"] = wide.Long < wide.Short
print(wide.round(3).to_string())
print(f"\nLong slope below Short slope in {int(wide.long_below_short.sum())} of 4 datasets; "
      f"mean Short {wide.Short.mean():.3f}, mean Long {wide.Long.mean():.3f}")
wu = slopes.pivot(index="dataset", columns="prior", values="slope_unweighted")
print(f"Unweighted slopes: Long below Short in {int((wu.Long < wu.Short).sum())} of 4")

**Result: reproduced.** The Long prior slope is below the Short prior slope in all 4 datasets (mean
0.716 against 0.898), weighted or unweighted.

### beh-bayes-model

> A Bayesian observer model with scalar measurement and production noise fit to the behaviour
> captures the bias and the variance of tp in all conditions (Figure 2C and 2D).

In [ ]:
models, rows, pred_rows = {}, [], []
for e, th, name in DS:
    g = good[(good.is_eye == e) & (good.theta == th)]
    m = rsg.BayesObserver.fit(g)
    models[name] = m
    obs_bias, pred_bias, obs_sd, pred_sd = [], [], [], []
    for short in [True, False]:
        ts = np.array(rsg.prior_ts(short), float)
        _, mean_tp, sd_tp = m.moments(ts, short)
        gg = g[g.is_short == short].groupby("ts").tp.agg(["mean", "std"]).loc[ts]
        obs_bias += list(gg["mean"] - ts)
        pred_bias += list(mean_tp - ts)
        obs_sd += list(gg["std"])
        pred_sd += list(sd_tp)
        for t, ob, pb, os_, ps in zip(ts, gg["mean"] - ts, mean_tp - ts, gg["std"], sd_tp):
            pred_rows.append({"dataset": name, "prior": "Short" if short else "Long", "ts": t,
                              "bias_obs": ob, "bias_model": pb, "sd_obs": os_, "sd_model": ps})
    ob, pb = np.array(obs_bias), np.array(pred_bias)
    r2 = 1 - np.sum((ob - pb) ** 2) / np.sum((ob - ob.mean()) ** 2)
    r_sd = stats.pearsonr(obs_sd, pred_sd)[0]
    rows.append({"dataset": name, "wm": m.wm, "wp": m.wp, "b_short": m.b[True], "b_long": m.b[False],
                 "converged": m.converged, "bias_R2": r2, "sd_r": r_sd})
fits = pd.DataFrame(rows)
print(fits.round(3).to_string(index=False))
print(f"\nBias R2 >= 0.8 in {(fits.bias_R2 >= 0.8).sum()} of 4; SD correlation > 0 in {(fits.sd_r > 0).sum()} of 4")
pred = pd.DataFrame(pred_rows)
print("\nObserved and model bias and SD (ms):")
print(pred.round(1).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for short, col in [(True, "tab:red"), (False, "tab:blue")]:
    p_ = pred[pred.prior == ("Short" if short else "Long")]
    axes[0].scatter(p_.bias_model, p_.bias_obs, color=col, s=15)
    axes[1].scatter(p_.sd_model, p_.sd_obs, color=col, s=15)
for ax, lab in zip(axes, ["bias (ms)", "SD of tp (ms)"]):
    lim = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lim, lim, "k:", lw=1)
    ax.set_xlabel("model " + lab)
    ax.set_ylabel("observed " + lab)
plt.tight_layout()
plt.show()
print("Bias: observed %.0f to %.0f, model %.0f to %.0f ms; SD: observed %.0f to %.0f, model %.0f to %.0f ms; all within the axes."
      % (pred.bias_obs.min(), pred.bias_obs.max(), pred.bias_model.min(), pred.bias_model.max(),
         pred.sd_obs.min(), pred.sd_obs.max(), pred.sd_model.min(), pred.sd_model.max()))

**Result: partially reproduced.** The observer model predicts the SD of tp with a positive correlation
in all 4 datasets (r = 0.28 to 0.91), but its predicted bias reaches R2 >= 0.8 in only 2 of 4
(R2 = 0.12, 0.67, 0.87 and 0.98). The two eye datasets, whose Short slopes exceed 1 (beh-slopes),
get very low measurement noise (wm = 0.013 and 0.025) and are the poorly fit ones. Every fit
converged. Table S3 was not available, so the fitted parameters are not compared with the paper's.

## Single units

### glm-prior-ts

> About 30% of neurons were modulated by elapsed time (27% in monkey H) and more than 60% were
> sensitive to the prior (65% in monkey H) (Figure 3E).

In [ ]:
def glm_rows(trs, ts_values):
    # One row per trial and time point t (ms after Ready) for which Set came after t (attrition).
    elapsed = ((trs.set_time - trs.ready_time) * 1000).to_numpy()
    cnt = np.stack([rsg.counts(spikes, units, trs.ready_time.to_numpy() + t / 1000, -rsg.GLM_WINDOW_MS, 0,
                               rsg.GLM_WINDOW_MS)[:, 0, :] for t in ts_values], axis=1)
    rows, ys = [], []
    for i, (_, r) in enumerate(trs.iterrows()):
        for k, t in enumerate(ts_values):
            if elapsed[i] >= t - 1e-6:
                rows.append({"t": k, "prior": float(not r.is_short), "eff": float(not r.is_eye), "dir": float(r.theta == 180)})
                ys.append(cnt[i, k])
    return pd.DataFrame(rows), np.array(ys)


def design(df, with_t=True, with_prior=True):
    cols = [np.ones(len(df)), df.eff.to_numpy(), df["dir"].to_numpy()]
    if with_prior:
        cols.append(df.prior.to_numpy())
    if with_t:
        cols += [(df.t == k).to_numpy().astype(float) for k in range(1, 5)]
    return np.column_stack(cols)


def fit(y, X):
    r = sm.GLM(y, X, family=sm.families.Poisson()).fit()
    return r.llf, X.shape[1]


def bic(llf, k, n):
    return -2 * llf + k * np.log(n)


res = []
df_all, Y_all = glm_rows(good, rsg.SHORT_TS)
per_prior = {short: glm_rows(good[good.is_short == short], rsg.prior_ts(short)) for short in [True, False]}
for j, u in enumerate(units):
    y = Y_all[:, j]
    l_full, k_full = fit(y, design(df_all))
    l_red, k_red = fit(y, design(df_all, with_prior=False))
    n = len(y)
    row = {"unit": int(S["unit_ids"][u]), "prior_dBIC": bic(l_red, k_red, n) - bic(l_full, k_full, n),
           "prior_LR_p": stats.chi2.sf(2 * (l_full - l_red), k_full - k_red)}
    for short, (dfp, Yp) in per_prior.items():
        yp = Yp[:, j]
        lf, kf = fit(yp, design(dfp, with_prior=False))
        lr, kr = fit(yp, design(dfp, with_t=False, with_prior=False))
        row[f"ts_dBIC_{'short' if short else 'long'}"] = bic(lr, kr, len(yp)) - bic(lf, kf, len(yp))
        row[f"ts_LR_p_{'short' if short else 'long'}"] = stats.chi2.sf(2 * (lf - lr), kf - kr)
    res.append(row)
glm = pd.DataFrame(res)
glm["prior_dep"] = glm.prior_dBIC > 0
glm["ts_dep"] = (glm.ts_dBIC_short > 0) | (glm.ts_dBIC_long > 0)


def wilson(k, n):
    lo, hi = sm.stats.proportion_confint(k, n, method="wilson")
    return f"{100 * k / n:.1f}% ({k} of {n}; 95% interval {100 * lo:.1f} to {100 * hi:.1f}%)"


print("Rows (trial x time point): all trials", len(df_all), "; Short", len(per_prior[True][0]), "; Long", len(per_prior[False][0]))
print("Prior-dependent (BIC):", wilson(int(glm.prior_dep.sum()), len(glm)), "; paper, H: 65%")
print("ts-dependent (BIC):   ", wilson(int(glm.ts_dep.sum()), len(glm)), "; paper, H: 27%")
print("Both:", int((glm.prior_dep & glm.ts_dep).sum()), "; prior only:", int((glm.prior_dep & ~glm.ts_dep).sum()),
      "; ts only:", int((~glm.prior_dep & glm.ts_dep).sum()), "; neither:", int((~glm.prior_dep & ~glm.ts_dep).sum()))
print("\nVariants, likelihood-ratio test in place of BIC:")
for a in [0.05, 0.01]:
    pd_ = (glm.prior_LR_p < a).sum()
    td_ = ((glm.ts_LR_p_short < a) | (glm.ts_LR_p_long < a)).sum()
    print(f"   p < {a}: prior {100 * pd_ / len(glm):.1f}%, ts {100 * td_ / len(glm):.1f}%")
print("ts-dependent in Short only:", int(((glm.ts_dBIC_short > 0) & ~(glm.ts_dBIC_long > 0)).sum()),
      "; Long only:", int((~(glm.ts_dBIC_short > 0) & (glm.ts_dBIC_long > 0)).sum()),
      "; both:", int(((glm.ts_dBIC_short > 0) & (glm.ts_dBIC_long > 0)).sum()))

**Result: not reproduced.** By the GLM and BIC rule of the Methods, 75.0% of units are prior-dependent
(36 of 48, 95% interval 61.2 to 85.1%; paper 65 +/- 6.5%) and 45.8% are ts-dependent (22 of 48,
interval 32.6 to 59.7%; paper 27 +/- 2.7%). Both fractions are above the paper's and outside
tolerance; the interval of the prior fraction includes 65%, that of the ts fraction does not
include 27%. With a likelihood-ratio test in place of BIC the fractions are higher still (prior
77.1 to 87.5%, ts 66.7 to 77.1%). The paper's fractions pool 536 units over 17 sessions; these 48
come from one session.

### dfr-overlap-support

> At the overlapping ts of 800 ms, firing rates differ between priors, with the largest difference
> during the support of the prior, between 480 and 800 ms after Ready (Figure 3D).

In [ ]:
rng_dfr = np.random.default_rng(3)
edges = np.arange(-320, 1440 + 1, 160)
centres = (edges[:-1] + edges[1:]) / 2


def rms_dfr(cnt, short):
    d = cnt[short].mean(0) - cnt[~short].mean(0)          # bins x units, counts per 160 ms
    return np.sqrt(np.mean((d * 1000 / 160) ** 2, axis=1))  # spikes/s


per_ds, nulls = {}, []
cnts = {}
for e, th, name in DS:
    g = good[(good.is_eye == e) & (good.theta == th) & (good.ts == 800)]
    cnts[name] = (rsg.counts(spikes, units, g.ready_time.to_numpy(), -320, 1440, 160), g.is_short.to_numpy())
    per_ds[name] = rms_dfr(*cnts[name])
mean_rms = np.mean(list(per_ds.values()), axis=0)
for _ in range(200):
    nulls.append(np.mean([rms_dfr(cnt, rng_dfr.permutation(lab)) for cnt, lab in cnts.values()], axis=0))
nulls = np.array(nulls)
tab = pd.DataFrame({"bin_start": edges[:-1], "bin_end": edges[1:], **{k: v for k, v in per_ds.items()},
                    "mean": mean_rms, "shuffle_mean": nulls.mean(0), "shuffle_95": np.percentile(nulls, 95, axis=0)})
print("RMS over units of the Short minus Long rate difference at ts = 800 ms (spikes/s), by 160 ms bin from Ready:")
print(tab.round(2).to_string(index=False))
k = int(np.argmax(mean_rms))
thr = np.percentile(nulls.max(1), 95)
print(f"\nLargest mean RMS: {mean_rms[k]:.2f} spikes/s in the bin {edges[k]} to {edges[k + 1]} ms after Ready; "
      f"95th percentile of the shuffled maximum: {thr:.2f}")
print("Largest bin per dataset:", {n: f"{edges[np.argmax(v)]}-{edges[np.argmax(v) + 1]}" for n, v in per_ds.items()})
print("Used trials at ts = 800 ms per dataset (Short, Long):",
      {n: (int(l.sum()), int((~l).sum())) for n, (c_, l) in cnts.items()})

In [ ]:
plt.figure(figsize=(6, 3.2))
for n, v in per_ds.items():
    plt.plot(centres, v, color="0.6", lw=1)
plt.plot(centres, mean_rms, "k", lw=2, label="mean of datasets")
plt.fill_between(centres, np.percentile(nulls, 5, axis=0), np.percentile(nulls, 95, axis=0), color="tab:green", alpha=.2, label="shuffle 5-95%")
plt.axvspan(480, 800, color="tab:red", alpha=.1, label="Short support")
plt.axvline(0, color="k", ls=":")
plt.axvline(800, color="k", ls="--")
plt.xlabel("time from Ready (ms); Set at 800")
plt.ylabel("RMS rate difference (spikes/s)")
plt.legend(fontsize=7)
plt.show()
print("Plotted values: %.2f to %.2f spikes/s; all within the axes." % (min(v.min() for v in per_ds.values()), max(v.max() for v in per_ds.values())))

**Result: reproduced.** At ts = 800 ms, the RMS over units of the Short minus Long rate difference,
averaged over datasets, is largest in the bin 480 to 640 ms after Ready (6.86 spikes/s), within the
support of the Short prior, and above the 95th percentile of the shuffled maximum (3.11). Before
Ready it is at the shuffle level (2.39 and 2.29 against 95th percentiles of 2.42 and 2.32). Per
dataset the largest bin is 480-640 (2 datasets), 640-800 and 800-960 ms; the mean is also high just
after Set (6.07 in 800 to 960 ms).

## Population trajectories

### pca-dimensionality

> Population activity is low dimensional; 3 to 4 principal components in the estimation epoch and
> 5 to 10 in the production epoch explain about 75% of the variance (Figure S3).

The PCA fitted here is used by the claims that follow.

In [ ]:
EST, PROD, rows = {}, {}, []
for e, th, name in DS:
    g = good[(good.is_eye == e) & (good.theta == th)]
    est = {}
    for short in [True, False]:
        gg = g[g.is_short == short]
        rate, ends, n = rsg.estimation_rates(spikes, units, gg, max(rsg.prior_ts(short)))
        est[short] = {"rate": rate, "t": ends, "n": n}
    pca_e = rsg.PCA([est[True]["rate"], est[False]["rate"]])
    prod = {}
    for short in [True, False]:
        for ts in rsg.prior_ts(short):
            gg = g[(g.is_short == short) & (g.ts == ts)]
            prod[(short, ts)] = {"rate": rsg.production_rates(spikes, units, gg), "min_tp": gg.tp.min(),
                                 "mean_tp": gg.tp.mean(), "n": len(gg)}
    pca_p = rsg.PCA([v["rate"] for v in prod.values()])
    EST[name] = {"pca": pca_e, **est}
    PROD[name] = {"pca": pca_p, "traj": prod}
    cum_e, cum_p = np.cumsum(pca_e.explained), np.cumsum(pca_p.explained)
    rows.append({"dataset": name, "est_PCs_75": pca_e.n, "prod_PCs_75": pca_p.n,
                 "est_PCs_70": int(np.searchsorted(cum_e, .7) + 1), "est_PCs_80": int(np.searchsorted(cum_e, .8) + 1),
                 "prod_PCs_70": int(np.searchsorted(cum_p, .7) + 1), "prod_PCs_80": int(np.searchsorted(cum_p, .8) + 1),
                 "est_var_3PC": cum_e[2], "prod_var_5PC": cum_p[4]})
dims = pd.DataFrame(rows)
print(dims.round(3).to_string(index=False))
n_ok = dims.est_PCs_75.between(3, 4).sum() + dims.prod_PCs_75.between(5, 11).sum()
print(f"\nIn range (estimation 3 or 4, production 5 to 11): {n_ok} of 8")
print("Shortest tp per ts, used to cut production trajectories (ms):",
      {f"{'S' if k[0] else 'L'}{k[1]}": round(min(PROD[n]['traj'][k]['min_tp'] for n in PROD)) for k in PROD[DS[0][2]]['traj']})

**Result: not reproduced.** Two components reach 75% of the variance in the estimation epoch in all 4
datasets (3 components explain 86 to 90%), and 2, 2, 5 and 4 in the production epoch, so 1 of the
8 values falls in the paper's ranges. Activity here is lower dimensional than the paper reports.
This session has 48 units; the paper's PCA pools up to 536 units recorded over 17 sessions, which
can add dimensions.

### encoding-r2

> Projections of estimation-epoch neural states onto the encoding axis match the Bayesian estimates
> from the model fitted to the behaviour (R2 = 0.993 for the Short prior, 0.996 for the Long prior,
> monkey H, Hand Left; Figure 4C).

In [ ]:
rng_axes = np.random.default_rng(4)
PROJ, rows = {}, []
for e, th, name in DS:
    E = EST[name]
    for short in [True, False]:
        lo, hi = min(rsg.prior_ts(short)), max(rsg.prior_ts(short))
        X = E["pca"](E[short]["rate"])
        t = E[short]["t"]
        u = rsg.encoding_axis(X, t, lo, hi)
        sel = (t >= lo - 1e-6) & (t <= hi + 1e-6)
        xu, tt, w = X[sel] @ u, t[sel], E[short]["n"][sel]
        te = models[name].moments(tt, short)[0]
        _, _, r2_b, rmse_b = rsg.weighted_fit(te, xu, w)
        _, slope_ts, r2_l, rmse_l = rsg.weighted_fit(tt, xu, w)
        rand = []
        for _ in range(1000):
            v = rng_axes.standard_normal(X.shape[1])
            rand.append(rsg.weighted_fit(te, X[sel] @ (v / np.linalg.norm(v)), w)[2])
        rand = np.array(rand)
        PROJ[(name, short)] = {"t": tt, "xu": xu, "w": w, "te": te, "u": u, "X": X, "tall": t}
        rows.append({"dataset": name, "prior": "Short" if short else "Long", "points": int(sel.sum()), "PCs": X.shape[1],
                     "R2_bayes": r2_b, "RMSE_bayes": rmse_b, "R2_linear": r2_l, "RMSE_linear": rmse_l,
                     "slope_ts": slope_ts, "random_R2_median": np.median(rand),
                     "random_R2_95": np.percentile(rand, 95), "frac_random_above": np.mean(rand >= r2_b)})
enc = pd.DataFrame(rows)
print(enc.round(4).to_string(index=False))
hl = enc[enc.dataset == "Hand Left"].set_index("prior").R2_bayes
mean_r2 = enc.groupby("prior").R2_bayes.mean()
print(f"\nReading (a), Hand Left: Short R2 = {hl['Short']:.4f}, Long R2 = {hl['Long']:.4f}")
print(f"Reading (b), mean of 4 datasets: Short R2 = {mean_r2['Short']:.4f}, Long R2 = {mean_r2['Long']:.4f}")
print("Tolerance: R2 >= 0.894 (Short) and >= 0.896 (Long)")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, (e, th, name) in zip(axes, DS):
    for short, col in [(True, "tab:red"), (False, "tab:blue")]:
        p_ = PROJ[(name, short)]
        a, b, _, _ = rsg.weighted_fit(p_["te"], p_["xu"], p_["w"])
        ax.plot(p_["t"], (p_["xu"] - a) / b, "o", color=col, ms=3)   # projections mapped onto te by the fit
        ax.plot(p_["t"], p_["te"], "-", color=col)
    ax.plot([450, 1250], [450, 1250], "k:", lw=1)
    ax.set_title(name)
    ax.set_xlabel("time from Ready = ts (ms)")
axes[0].set_ylabel("te (line) and mapped Xu (dots), ms")
plt.tight_layout()
plt.show()
vals = np.concatenate([(PROJ[k]["xu"] - rsg.weighted_fit(PROJ[k]["te"], PROJ[k]["xu"], PROJ[k]["w"])[0]) / rsg.weighted_fit(PROJ[k]["te"], PROJ[k]["xu"], PROJ[k]["w"])[1] for k in PROJ])
print("Mapped projections: %.0f to %.0f ms; all within the axes." % (vals.min(), vals.max()))

**Result: reproduced.** Under reading (a), Hand Left, R2 = 0.998 (Short) and 0.996 (Long); under
reading (b), the mean of the 4 datasets, 0.959 and 0.989. Both readings are within tolerance. The
estimation trajectories here have only 2 PCs, and projections on random unit vectors in that space
also fit te well (median R2 0.73 to 0.98 by dataset and prior). The R2 on u exceeds that of 95% of
random axes in 3 of 8 dataset-priors (Hand Right Long, both Hand Left priors), so in this session the
fit is specific to u only for those three.

### encoding-bayes-beats-linear

> Neural projections are better explained by the Bayesian model than by a linear function of ts
> with the same number of parameters, for both priors (signed-rank test on RMSE, p = 0.008 for each
> prior; Figure 4D top).

In [ ]:
enc["dRMSE_bayes_minus_linear"] = enc.RMSE_bayes - enc.RMSE_linear
print(enc[["dataset", "prior", "RMSE_bayes", "RMSE_linear", "dRMSE_bayes_minus_linear"]].round(3).to_string(index=False))
for prior, g in enc.groupby("prior", sort=False):
    print(f"{prior}: Bayesian lower in {(g.dRMSE_bayes_minus_linear < 0).sum()} of 4; "
          f"signed-rank p = {stats.wilcoxon(g.RMSE_bayes, g.RMSE_linear).pvalue:.3f} (smallest possible with 4: 0.125)")

**Result: partially reproduced.** The Bayesian model has the lower RMSE in all 4 datasets for the
Long prior and in 3 of 4 for the Short prior (Hand Right Short is the exception, 0.99 against 0.80).
The differences are small, 0.045 to 0.573 in units of Xu (spikes/s).

### short-in-long-control

> In the Long prior during the period matched to the support of the Short prior ("Short In Long"),
> the Bayesian model is not better than the linear one (signed-rank test, p = 0.74).

In [ ]:
rows = []
for e, th, name in DS:
    p_ = PROJ[(name, False)]
    X, t = p_["X"], p_["tall"]
    u = rsg.encoding_axis(X, t, 480, 800)
    sel = (t >= 480 - 1e-6) & (t <= 800 + 1e-6)
    xu, tt = X[sel] @ u, t[sel]
    w = EST[name][False]["n"][sel]
    te = models[name].moments(tt, True)[0]
    rows.append({"dataset": name, "RMSE_bayes": rsg.weighted_fit(te, xu, w)[3], "RMSE_linear": rsg.weighted_fit(tt, xu, w)[3],
                 "R2_bayes": rsg.weighted_fit(te, xu, w)[2], "R2_linear": rsg.weighted_fit(tt, xu, w)[2]})
sil = pd.DataFrame(rows)
sil["dRMSE"] = sil.RMSE_bayes - sil.RMSE_linear
print(sil.round(4).to_string(index=False))
p_sil = stats.wilcoxon(sil.RMSE_bayes, sil.RMSE_linear).pvalue
print(f"\nShort In Long: Bayesian lower in {(sil.dRMSE < 0).sum()} of 4; median dRMSE {sil.dRMSE.median():.3f}; "
      f"signed-rank p = {p_sil:.3f} (smallest possible with 4 datasets: 0.125)")
sh = enc[enc.prior == "Short"]
print(f"For comparison, Short prior: median dRMSE {sh.dRMSE_bayes_minus_linear.median():.3f}; "
      f"relative dRMSE (dRMSE / RMSE_linear) Short In Long {np.median(sil.dRMSE / sil.RMSE_linear):.3f}, "
      f"Short {np.median(sh.dRMSE_bayes_minus_linear / sh.RMSE_linear):.3f}")

**Result: reproduced by the paper's test, which cannot detect an effect here.** The signed-rank test
over the 4 datasets gives p = 0.25; with 4 datasets the smallest attainable p is 0.125, so it could
not have found an effect. The effect sizes do not show the contrast the paper describes: in the
control period the Bayesian model is better in 3 of 4 datasets, with a median relative RMSE
difference of -0.089, at least as large as in the Short prior itself (-0.060).

### neural-long-more-bias

> Neural projections show more bias for the Long prior than for the Short prior, as a smaller
> regression slope of Xu on ts (signed-rank test, p = 0.008; Figure 4D bottom).

In [ ]:
sl = enc.pivot(index="dataset", columns="prior", values="slope_ts")
sl["long_below_short"] = sl.Long < sl.Short
print("Slope of Xu on ts (spikes/s per ms, shared PC space of each dataset):")
print(sl.round(4).to_string())
print(f"\nLong below Short in {int(sl.long_below_short.sum())} of 4 datasets; ratio Long / Short: "
      + ", ".join(f"{r:.2f}" for r in sl.Long / sl.Short))

**Result: reproduced.** The slope of Xu on ts is smaller for the Long prior in all 4 datasets (ratios
Long / Short 0.55, 0.63, 0.34 and 0.97); the Hand Right difference is small.

### speed-vs-projection

> Larger projections on the encoding axis at Set are followed by slower neural trajectories in the
> production epoch (Pearson correlation, rho Short = -0.74, rho Long = -0.51; Figure 4E).

In [ ]:
rows = []
for e, th, name in DS:
    P = PROD[name]
    for (short, ts), v in P["traj"].items():
        states = P["pca"](v["rate"])
        p_ = PROJ[(name, short)]
        rows.append({"dataset": name, "prior": "Short" if short else "Long", "ts": ts,
                     "xu_at_set": p_["xu"][np.argmin(abs(p_["t"] - ts))], "speed": rsg.step_speed(states),
                     "duration_ms": len(states) * rsg.BIN_MS, "mean_tp": v["mean_tp"], "trials": v["n"]})
spd = pd.DataFrame(rows)
spd["speed_per_tp"] = spd.speed / spd.mean_tp
print(spd.round(3).to_string(index=False))
for prior in ["Short", "Long"]:
    g = spd[spd.prior == prior]
    r, p = stats.pearsonr(g.xu_at_set, g.speed)
    rb, pb = stats.pearsonr(g.xu_at_set, g.speed_per_tp)
    z = g.groupby("dataset").xu_at_set.transform(lambda x: (x - x.mean()) / x.std())
    rz, pz = stats.pearsonr(z, g.speed)
    within = [stats.pearsonr(h.xu_at_set, h.speed)[0] for _, h in g.groupby("dataset")]
    print(f"{prior}: rho = {r:.3f} (p = {p:.2g}); variant speed / mean tp: rho = {rb:.3f} (p = {pb:.2g}); "
          f"Xu z-scored per dataset: rho = {rz:.3f} (p = {pz:.2g}); within datasets: " + ", ".join(f"{x:.2f}" for x in within))

**Result: partially reproduced.** Larger projections at Set are followed by slower production
trajectories in both priors, more strongly than the paper reports: rho = -0.941 (Short, p = 6.2e-10)
and -0.878 (Long, p = 3.7e-07), outside the tolerances around -0.74 and -0.51. Within each dataset
the correlations are -0.75 to -1.00, and with Xu z-scored per dataset they are -0.759 and -0.779.

### initial-speed-biased

> The initial speed of production trajectories (Set to Set + 400 ms) is biased toward the middle of
> the prior; its dependence on ts is shallower than the unbiased profile proportional to 1/ts
> (signed-rank test, p < 1e-3 over conditions and animals; Figure 4F).

In [ ]:
rows, dropped = [], 0
for e, th, name in DS:
    g = good[(good.is_eye == e) & (good.theta == th)]
    for short in [True, False]:
        ts_v = np.array(rsg.prior_ts(short), float)
        sp = []
        for ts in ts_v:
            gg = g[(g.is_short == short) & (g.ts == ts)]
            dropped += int((gg.tp < 400).sum())
            gg = gg[gg.tp >= 400]
            sp.append(rsg.step_speed(PROD[name]["pca"](rsg.production_rates(spikes, units, gg, 400))))
        sp = np.array(sp)
        unbiased = sp[2] * ts_v[2] / ts_v
        rows.append({"dataset": name, "prior": "Short" if short else "Long", **{f"speed_{int(t)}": s for t, s in zip(ts_v, sp)},
                     "slope_measured": np.polyfit(ts_v, sp, 1)[0], "slope_unbiased": np.polyfit(ts_v, unbiased, 1)[0]})
ini = pd.DataFrame(rows)
ini["measured_minus_unbiased"] = ini.slope_measured - ini.slope_unbiased
print(ini.round(5).to_string(index=False))
w = stats.wilcoxon(ini.slope_measured, ini.slope_unbiased)
print(f"\nMeasured shallower than unbiased in {(ini.measured_minus_unbiased > 0).sum()} of 8; "
      f"two-sided signed-rank p = {w.pvalue:.4f}; trials dropped for tp < 400 ms: {dropped}")
print("Measured slope negative in", int((ini.slope_measured < 0).sum()), "of 8")

**Result: reproduced.** The initial speed falls with ts in all 8 dataset-priors, and its slope is
shallower than the unbiased 1/ts profile in 7 of 8 (two-sided signed-rank p = 0.039). Eye Right,
Long is the exception. No trial was dropped for tp < 400 ms.

### speed-predicts-tp

> The speed of neural trajectories in the production epoch predicts the mean tp (Pearson
> correlation, rho Short = -0.58, rho Long = -0.52; Figure 4G).

In [ ]:
for prior in ["Short", "Long"]:
    g = spd[spd.prior == prior]
    r, p = stats.pearsonr(g.speed, g.mean_tp)
    rb, pb = stats.pearsonr(g.speed_per_tp, g.mean_tp)
    within = [stats.pearsonr(h.speed, h.mean_tp)[0] for _, h in g.groupby("dataset")]
    print(f"{prior}: rho(speed, mean tp) = {r:.3f} (p = {p:.2g}), 20 points; variant speed / mean tp: {rb:.3f} (p = {pb:.2g}); "
          "within datasets: " + ", ".join(f"{x:.2f}" for x in within))

# Variant: the Neural Latents Benchmark's measure, trial by trial within prior, effector and
# direction; single-trial rates smoothed as above and projected on the production PCs, from Set
# to the trial's own Go.
rows = []
for e, th, name in DS:
    g = good[(good.is_eye == e) & (good.theta == th)]
    for short in [True, False]:
        gg = g[g.is_short == short]
        c_ = rsg.counts(spikes, units, gg.set_time.to_numpy(), 0, 2400)
        sp = []
        for i, tp in enumerate(gg.tp.to_numpy()):
            nb = int(tp // rsg.BIN_MS)
            sp.append(rsg.step_speed(PROD[name]["pca"](rsg.smooth(c_[i, :nb] * 1000 / rsg.BIN_MS))))
        r, p = stats.pearsonr(sp, gg.tp)
        rows.append({"dataset": name, "prior": "Short" if short else "Long", "trials": len(gg), "r": r, "p": p})
nlb = pd.DataFrame(rows)
print("\nTrial-by-trial speed against tp (NLB-style):")
print(nlb.round(3).to_string(index=False))
print(f"Mean r over the 8 conditions: {nlb.r.mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for prior, col in [("Short", "tab:red"), ("Long", "tab:blue")]:
    g = spd[spd.prior == prior]
    for n, h in g.groupby("dataset"):
        axes[0].plot(h.xu_at_set, h.speed, "o-", color=col, ms=3, lw=.7)
        axes[1].plot(h.speed, h.mean_tp, "o-", color=col, ms=3, lw=.7)
axes[0].set_xlabel("Xu at Set (spikes/s)")
axes[0].set_ylabel("speed (spikes/s per 20 ms)")
axes[1].set_xlabel("speed (spikes/s per 20 ms)")
axes[1].set_ylabel("mean tp (ms)")
plt.tight_layout()
plt.show()
print("Xu %.1f to %.1f; speed %.2f to %.2f; mean tp %.0f to %.0f; all within the axes."
      % (spd.xu_at_set.min(), spd.xu_at_set.max(), spd.speed.min(), spd.speed.max(), spd.mean_tp.min(), spd.mean_tp.max()))

**Result: partially reproduced.** Faster production trajectories go with shorter mean tp in both
priors: rho = -0.733 (Short, p = 0.00024) and -0.744 (Long, p = 0.00017) over 20 points, stronger
than the paper's -0.58 and -0.52 and outside tolerance. Within datasets the correlations are -0.80
to -0.99. The benchmark's trial-by-trial variant gives r = -0.195 to -0.740 (mean -0.519), with 7 of
8 conditions at p < 0.05.

## Alternative mechanisms

### alt-speed-unmodulated

> Instantaneous speed of neural trajectories in the full neural space is not modulated during the
> support of each prior (signed-rank test for zero regression slopes, p = 0.64 for Short, p = 0.08
> for Long; Figure 5B).

In [ ]:
rng_boot = np.random.default_rng(5)


def support_slope(rate, ends, short):
    lo, hi = min(rsg.prior_ts(short)), max(rsg.prior_ts(short))
    sp = np.linalg.norm(np.diff(rate, axis=0), axis=1)
    t = ends[1:]
    sel = (t > lo) & (t <= hi + 1e-6)
    return np.polyfit(t[sel], sp[sel], 1)[0] * 100, sp[sel].mean()   # per 100 ms


rows = []
for e, th, name in DS:
    g = good[(good.is_eye == e) & (good.theta == th)]
    for short in [True, False]:
        gg = g[g.is_short == short]
        t_max = max(rsg.prior_ts(short))
        c_ = rsg.counts(spikes, units, gg.ready_time.to_numpy(), 0, t_max)
        el = ((gg.set_time - gg.ready_time) * 1000).to_numpy()
        rate, ends, _ = rsg.attrition_rates(c_, el)
        slope, mean_sp = support_slope(rate, ends, short)
        boots = []
        for _ in range(200):
            i = rng_boot.integers(0, len(gg), len(gg))
            r_, e_, _ = rsg.attrition_rates(c_[i], el[i])
            boots.append(support_slope(r_, e_, short)[0])
        rows.append({"dataset": name, "prior": "Short" if short else "Long", "mean_speed": mean_sp, "slope_per_100ms": slope,
                     "slope_rel_pct": 100 * slope / mean_sp, "boot_low": np.percentile(boots, 2.5), "boot_high": np.percentile(boots, 97.5)})
asp = pd.DataFrame(rows)
print("Speed in the full space of", len(units), "units (spikes/s per 20 ms step); slope per 100 ms, with 95% bootstrap interval over trials:")
print(asp.round(3).to_string(index=False))
for prior, g in asp.groupby("prior", sort=False):
    print(f"{prior}: signed-rank p = {stats.wilcoxon(g.slope_per_100ms).pvalue:.3f} (smallest possible with 4: 0.125); "
          f"slopes {', '.join(f'{x:+.3f}' for x in g.slope_per_100ms)}; intervals excluding 0: "
          f"{int(((g.boot_low > 0) | (g.boot_high < 0)).sum())} of 4")

**Result: reproduced by the paper's test, which cannot detect an effect here; the data show the
speed is modulated.** The signed-rank test over 4 datasets gives p = 0.125 for each prior, the
smallest value it can give. All 8 slopes are positive, speed rising by 13 to 24% of its mean per 100
ms during the support, and all 8 bootstrap intervals exclude zero. So in this session the speed in
the full neural space increases during the support of each prior; the claim of no modulation holds
only in the sense that the paper's test, at this sample size, cannot reject it. The paper's test
over 8 datasets gave p = 0.08 for Long.

### alt-transient-parallel

> In the first 200 ms after Set, trajectories for different ts run parallel; their distance to the
> middle ts trajectory does not change (signed-rank test for zero slopes, p = 0.92 for the shortest
> ts, p = 0.13 for the longest; Figure 5D).

In [ ]:
rows = []
for e, th, name in DS:
    g = good[(good.is_eye == e) & (good.theta == th)]
    traj = {}
    for short in [True, False]:
        for ts in rsg.prior_ts(short):
            gg = g[(g.is_short == short) & (g.ts == ts)]
            traj[(short, ts)] = rsg.aligned_rates(spikes, units, gg.set_time.to_numpy(), -100, 300)[5:15]  # Set to Set + 200 ms
    pca_t = rsg.PCA(list(traj.values()))
    t = np.arange(1, 11) * rsg.BIN_MS
    for short in [True, False]:
        ts_v = rsg.prior_ts(short)
        ref = pca_t(traj[(short, ts_v[2])])
        for which, ts in [("shortest", ts_v[0]), ("longest", ts_v[-1])]:
            dist = np.linalg.norm(pca_t(traj[(short, ts)]) - ref, axis=1)
            rows.append({"dataset": name, "prior": "Short" if short else "Long", "ts": which, "PCs": pca_t.n,
                         "mean_distance": dist.mean(), "slope_per_100ms": np.polyfit(t, dist, 1)[0] * 100})
tr_ = pd.DataFrame(rows)
tr_["slope_rel_pct"] = 100 * tr_.slope_per_100ms / tr_.mean_distance
print(tr_.round(3).to_string(index=False))
for which, g in tr_.groupby("ts"):
    w = stats.wilcoxon(g.slope_per_100ms)
    print(f"{which} ts: signed-rank p = {w.pvalue:.3f}; positive slopes {(g.slope_per_100ms > 0).sum()} of 8; "
          f"median relative slope {g.slope_rel_pct.median():+.1f}% per 100 ms")

**Result: not reproduced.** For the longest ts, the distance to the middle trajectory changes in the
first 200 ms after Set: 6 of 8 slopes are negative (median -15.3% of the mean distance per 100 ms),
signed-rank p = 0.039. For the shortest ts p = 0.078 (6 of 8 positive, median +26.7% per 100 ms).
The PCA kept 1 to 4 components, and in Eye Left a single component, where the distance is the
absolute difference along one axis.

### alt-threshold-monotonic

> Aligned to the response, distances between trajectories for different ts decrease steadily up to
> movement initiation, without the converging then diverging pattern a ts-dependent threshold would
> produce (Figure 5F).

In [ ]:
def go_curves(kept):
    rows, curves = [], {}
    for e, th, name in DS:
        g = good[(good.is_eye == e) & (good.theta == th)]
        traj = {}
        for short in [True, False]:
            for ts in rsg.prior_ts(short):
                gg = g[(g.is_short == short) & (g.ts == ts)]
                traj[(short, ts)] = rsg.aligned_rates(spikes, units, gg.go_time.to_numpy(), -900, 100)[5:45]  # Go - 800 to Go
        pca_g = rsg.PCA(list(traj.values()), kept) if kept else (lambda x: x)
        t = np.arange(-39, 1) * rsg.BIN_MS   # bin end times relative to Go
        for short in [True, False]:
            ts_v = rsg.prior_ts(short)
            ref = pca_g(traj[(short, ts_v[2])])
            for ts in ts_v[:2] + ts_v[3:]:
                dist = np.linalg.norm(pca_g(traj[(short, ts)]) - ref, axis=1)
                curves[(name, short, ts)] = dist
                rows.append({"dataset": name, "prior": "Short" if short else "Long", "ts": ts,
                             "PCs": getattr(pca_g, "n", len(units)), "t_min": t[np.argmin(dist)],
                             "d_at_go_over_min": dist[-1] / dist.min(), "d_start": dist[0], "d_go": dist[-1]})
    return pd.DataFrame(rows), curves, t


thr_tab, thr_curves, t_go = go_curves(0.60)
print(thr_tab.round(3).to_string(index=False))
print(f"\nMain (60% of variance): median time of the minimum {thr_tab.t_min.median():.0f} ms (bin end, relative to Go); "
      f"minimum in the last 40 ms in {(thr_tab.t_min >= -20).sum()} of {len(thr_tab)} curves; in the last 200 ms in "
      f"{(thr_tab.t_min >= -180).sum()}; distance at Go below that 800 ms earlier in {(thr_tab.d_go < thr_tab.d_start).sum()}")
for kept, lab in [(0.75, "75% of variance"), (None, "full space")]:
    v, _, _ = go_curves(kept)
    print(f"Variant, {lab}: median time of the minimum {v.t_min.median():.0f} ms; in the last 40 ms {(v.t_min >= -20).sum()} of {len(v)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for (name, short, ts), d in thr_curves.items():
    axes[0 if short else 1].plot(t_go, d, color="tab:red" if short else "tab:blue", lw=.6, alpha=.6)
for ax, lab in zip(axes, ["Short", "Long"]):
    ax.set_title(lab)
    ax.set_xlabel("time from Go (ms)")
axes[0].set_ylabel("distance to middle ts (spikes/s)")
plt.tight_layout()
plt.show()
allv = np.concatenate(list(thr_curves.values()))
print("Distances %.1f to %.1f; all within the axes." % (allv.min(), allv.max()))

**Result: not reproduced.** With the components that reach 60% of the variance (1 to 3 per dataset),
the median time of the minimum distance is 210 ms before Go; the minimum is in the last 40 ms in 6
of 32 curves and in the last 200 ms in 14. The distance at Go is below that 800 ms earlier in 26 of
32 curves, so distances do fall overall. The result depends on the number of components: with 75%
of the variance the median minimum is 80 ms before Go, and in the full space of 48 units 40 ms
before Go, which would meet the criterion. With 1 component (Eye datasets) the distance can reach
zero where trajectories cross, which moves the minima.

## Single-trial trajectories (GPFA)

### gpfa-variance-inverted-u

> The variance across single trials of the GPFA neural state at Set projected on the encoding axis
> has an inverted-U profile over ts, lower at the extremes of each prior (one-tailed signed-rank test
> on quadratic coefficients, p = 0.0273, monkey H; Figure 6C).

In [ ]:
rng_gpfa = np.random.default_rng(6)
t_gpfa = time.time()
GP, rows = {}, []
for e, th, name in DS:
    g = good[(good.is_eye == e) & (good.theta == th)]
    sts = rsg.estimation_spiketrains(spikes, units, g)
    model, dim, scores = rsg.fit_gpfa(sts, rng_gpfa)
    lat = rsg.gpfa_transform(model, sts)
    GP[name] = {"lat": lat, "trials": g, "dim": dim}
    print(f"{name}: {len(g)} trials, latent dimensionality {dim} (cross-validated log-likelihood "
          + ", ".join(f"{k}: {v:.0f}" for k, v in scores.items()) + ")")
    for short in [True, False]:
        mask = g.is_short.to_numpy() == short
        L = [lat[i] for i in np.flatnonzero(mask)]
        nb = max(x.shape[1] for x in L)
        mean = np.array([np.mean([x[:, k] for x in L if x.shape[1] > k], axis=0) for k in range(nb)])
        tt = np.arange(1, nb + 1) * rsg.BIN_MS
        ts_v = rsg.prior_ts(short)
        u = rsg.encoding_axis(mean, tt, min(ts_v), max(ts_v))
        xu = np.array([x[:, -1] @ u for x in L])
        ts_tr = g.ts.to_numpy()[mask]
        z = (xu - xu.mean()) / xu.std()
        var = np.array([z[ts_tr == t].var(ddof=1) for t in ts_v])
        mxu = np.array([xu[ts_tr == t].mean() for t in ts_v])
        rows.append({"dataset": name, "prior": "Short" if short else "Long", **{f"var_{t}": v for t, v in zip(ts_v, var)},
                     "quad_coef": np.polyfit(np.array(ts_v, float), var, 2)[0],
                     "mean_xu": mxu, "slope_xu": np.polyfit(np.array(ts_v, float), mxu, 1)[0]})
gp = pd.DataFrame(rows)
print(f"GPFA fits took {time.time() - t_gpfa:.0f} s")
print(gp.drop(columns=["mean_xu"]).to_string(index=False, float_format=lambda x: f"{x:.3g}"))
w = stats.wilcoxon(gp.quad_coef, alternative="less")
print(f"\nQuadratic coefficient negative in {(gp.quad_coef < 0).sum()} of 8; one-tailed signed-rank p = {w.pvalue:.4f}")

In [ ]:
var_cols = {True: [f"var_{t}" for t in rsg.SHORT_TS], False: [f"var_{t}" for t in rsg.LONG_TS]}
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2), sharey=True)
allv = []
for ax, short in zip(axes, [True, False]):
    for _, r in gp[gp.prior == ("Short" if short else "Long")].iterrows():
        v = r[var_cols[short]].to_numpy(float)
        allv += list(v)
        ax.plot(rsg.prior_ts(short), v, "o-", color="tab:red" if short else "tab:blue", lw=.8, ms=3)
    ax.set_xlabel("ts (ms)")
axes[0].set_ylabel("variance of z-scored Xu")
plt.tight_layout()
plt.show()
print("Variances %.2f to %.2f; all within the axes." % (min(allv), max(allv)))

**Result: not reproduced.** The quadratic coefficient of variance on ts is negative in 5 of 8
dataset-priors; the one-tailed signed-rank test gives p = 0.125 (paper 0.0273). Cross-validation
chose 8 latent dimensions in all 4 datasets, the larger of the two candidates.

### gpfa-mean-sigmoid

> The mean of the single-trial GPFA projections is sigmoidal (signed-rank test on increments,
> p = 0.0078) and has a higher slope for the Short prior (one-tailed signed-rank, p = 0.0625;
> Figure 6D).

In [ ]:
inc_g = pd.DataFrame([{"dataset": r.dataset, "prior": r.prior, **dict(zip(["extreme", "middle"], rsg.increments(r.mean_xu)))}
                      for _, r in gp.iterrows()])
print(inc_g.round(4).to_string(index=False))
w = stats.wilcoxon(inc_g.middle, inc_g.extreme)
print(f"\nMiddle increment above extreme in {(inc_g.middle > inc_g.extreme).sum()} of 8; two-sided signed-rank p = {w.pvalue:.4f}")
sg = gp.pivot(index="dataset", columns="prior", values="slope_xu")
print("\nSlope of mean Xu on ts (latent units per ms):")
print(sg.round(5).to_string())
print(f"Short slope above Long in {int((sg.Short > sg.Long).sum())} of 4; one-tailed signed-rank p = "
      f"{stats.wilcoxon(sg.Short, sg.Long, alternative='greater').pvalue:.4f}")
print("Mean Xu rises with ts (positive slope) in", int((gp.slope_xu > 0).sum()), "of 8")
print(f"\nTotal run time {time.time() - started:.0f} s")

**Result: reproduced.** The mean single-trial projection rises with ts in all 8 dataset-priors, its
middle increments exceed those at the extremes in 8 of 8 (two-sided signed-rank p = 0.0078, the
paper's value), and its slope is higher for the Short prior in 4 of 4 datasets (one-tailed p =
0.0625, the paper's value).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| beh-slopes | Not reproduced (6 of 8 slopes between 0 and 1) |
| beh-overlap-prior | Not reproduced (both biases positive, p = 0.073) |
| beh-sigmoid-increments | Reproduced (7 of 8, p = 0.0156) |
| beh-long-more-bias | Reproduced (4 of 4) |
| beh-bayes-model | Partially reproduced (SD in 4 of 4, bias R2 >= 0.8 in 2 of 4) |
| glm-prior-ts | Not reproduced (75.0% and 45.8% against 65% and 27%) |
| dfr-overlap-support | Reproduced (largest at 480 to 640 ms) |
| pca-dimensionality | Not reproduced (2 PCs in the estimation epoch) |
| encoding-r2 | Reproduced (0.998 and 0.996, Hand Left), not specific to u in 5 of 8 |
| encoding-bayes-beats-linear | Partially reproduced (Long 4 of 4, Short 3 of 4) |
| short-in-long-control | Reproduced by the paper's test, which cannot detect an effect with 4 datasets |
| neural-long-more-bias | Reproduced (4 of 4) |
| speed-vs-projection | Partially reproduced (rho -0.94 and -0.88, stronger than reported) |
| initial-speed-biased | Reproduced (7 of 8, p = 0.039) |
| speed-predicts-tp | Partially reproduced (rho -0.73 and -0.74, stronger than reported) |
| alt-speed-unmodulated | Reproduced by the paper's test, which cannot detect an effect; all 8 slopes positive |
| alt-transient-parallel | Not reproduced (longest ts, p = 0.039) |
| alt-threshold-monotonic | Not reproduced (median minimum 210 ms before Go with 60% of variance) |
| gpfa-variance-inverted-u | Not reproduced (5 of 8, p = 0.125) |
| gpfa-mean-sigmoid | Reproduced (p = 0.0078 and 0.0625, the paper's values) |
| rnn-curved-manifold, rnn-perturbation | Blocked: model results, no data in the dandiset |
| monkey-g | Blocked: data missing |

How the analysis differs from the paper's. All results come from one session of monkey H, one of
the 29 sessions of the paper, with 48 units where the paper pools 536 (H) and 636 (G) units over
sessions; a "dataset" here is one effector and direction in this session (4, against the paper's
8). The paper's code was not available, so smoothing, the encoding axis, speed, the GLM design with
attrition and the Bayesian observer fit follow the Methods as read and the choices stated in the
claims file. The paper's signed-rank tests over 8 datasets become tests over 4, which cannot reach
p < 0.05; two claims of absence are judged on such tests. The GPFA latent dimensionality was chosen
from 4 and 8 only, with 100 EM iterations. The outlier flag is the file's.

What was found about the dandiset. It holds one session of one of the paper's two monkeys, in the
form prepared for the Neural Latents Benchmark: 283 of the session's trials are in a test file
without task variables and without 14 of the 54 units. The 48 units that pass the Methods' rate rule
match the paper's single-session GPFA count for monkey H, consistent with this being that session.
The `iti` description gives the priors' inter-trial intervals the wrong way round, `tp` is a median
18 ms longer than `go_time - set_time`, the `is_outlier` flag is not reproduced by a refit of the
Methods' mixture model (10 of 46 flagged), and units have no single-unit or multi-unit label and no
electrode, whose table has no positions.